# CASO: Detección de fraudes - Creación de variables con LLM

En la siguiente actividad tendrá que explorar la data proporcionada, crear las variables propuestas usando LLM y proponer otras variables adicionales.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# SECCIÓN 1: Importar datos y exploración inicial

## Ejercicio 1
1a. Carge los datos con el nombre de df (Archivo: df_caso_fraude.csv)

1b. ¿Qué observa sobre las variables del dataset?

In [ ]:
# COMPLETAR 1a


In [ ]:
df.info()

COMPLETAR

1b.

## Ejercicio 2
Evalue la distribución de la variable target (is_fraud) ¿Esta balanceado o desbalanceado?

Pista: Puedes usar .value_counts()  o incluso hacer un grafico para ver la distribución

In [ ]:
# COMPLETAR


Distribución de transaction_type

In [ ]:
# Se observa el porcentaje que representa cada tipo de transacción
df["transaction_type"].value_counts(normalize=True)

Tasa de fraude por transaction_type.

In [ ]:
fraud_rate_by_type = (
    df.groupby("transaction_type")["is_fraud"]
      .mean()
      .sort_values(ascending=False)
)
fraud_rate_by_type

## Ejercicio 3
Según el resultado anterior, ¿qué tipo de transacción parece más riesgosa?

COMPLETAR




# SECCIÓN 2: Modelo base sin GenAI

Se construye un modelo base (LogisticRegression o RandomForest) que use como features:

*   time_since_login_min
*   transaction_amount
*   is_first_transaction
*   user_tenure_months
*   transaction_type (one-hot)

Se separo train/test (30% test, estratificado).

Se calcula AUC y recall para is_fraud = 1.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, recall_score

features_base = [
    "time_since_login_min",
    "transaction_amount",
    "is_first_transaction",
    "user_tenure_months",
    "transaction_type"
]

X = df[features_base]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

numeric_features = [
    "time_since_login_min",
    "transaction_amount",
    "is_first_transaction",
    "user_tenure_months"
]
cat_features = ["transaction_type"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

clf_base = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

clf_base.fit(X_train, y_train)

y_proba_base = clf_base.predict_proba(X_test)[:, 1]
y_pred_base = (y_proba_base >= 0.5).astype(int)

print("AUC base:", roc_auc_score(y_test, y_proba_base))
print("Recall base:", recall_score(y_test, y_pred_base))


## Ejercicio 4
En base al resultado anterior de AUC y Recall. ¿Qué tan buenos son los resultados para este caso de detección de fraudes?

COMPLETAR

# SECCIÓN 3: Diseñar y generación de variables con LLM

## Ejercicio 5

Define, en tus palabras, qué significa cada una de estas variables propuestas:

RISK_SEGMENT

ALERT_LEVEL

RECOMMENDED_ACTION



COMPLETAR

## Ejercicio 6
¿De que forma crees que un analista humano podría usar
estas variables del dataframe para generar las variables propuestas?:

Variables del dataframe
* time_since_login_min
* transaction_amount
* transaction_type
* is_first_transaction
* user_tenure_months
* predicted_fraud_proba

COMPLETAR

In [ ]:
# Importando librerias
import os
import json
import getpass
import google.genai as genai
import textwrap

## Ejercicio 7
Agrege el codigo necesario para ingresar la API KEY DE GOOGLE para usar un modelo de gemini

Debe de ingresarse con el nombre de GOOGLE_API_KEY

In [ ]:
# COMPLETAR


Diseñamos un prompt instructivo para que el LLM reciba el perfil de una transacción y devuelva:

RISK_SEGMENT (texto corto)

ALERT_LEVEL (1, 2, 3)

RECOMMENDED_ACTION (BLOCK, REVIEW, ALLOW)


Se implemento el prompt en código usando PromptTemplate de LangChain donde se indica que el resultado a devolver es un diccionario para cada registro

In [ ]:
genai.configure(api_key=os.environ["GOOGLE_API_KEY"]) # Using configure

model = genai.GenerativeModel("gemini-2.5-flash")


def build_batch_prompt(df_batch):
    lineas = []
    for i, row in df_batch.reset_index(drop=True).iterrows():
        idx = i + 1
        lineas.append(
            f'{idx}) monto={row["transaction_amount"]}, '
            f'tipo="{row["transaction_type"]}", '
            f'min_desde_login={row["time_since_login_min"]}, '
            f'primera_tx={row["is_first_transaction"]}, '
            f'antiguedad_meses={row["user_tenure_months"]}, '
            f'proba_modelo={row["predicted_fraud_proba"]}'
        )
    clientes_txt = "\n".join(lineas)

    prompt = f"""
Eres un analista de fraude en un banco digital.

Para cada transacción numerada debes:
- Definir RISK_SEGMENT (texto corto),
- Asignar ALERT_LEVEL (1=bajo, 2=medio, 3=alto),
- Recomendar RECOMMENDED_ACTION (BLOCK, REVIEW, ALLOW).

Transacciones:
{clientes_txt}

Responde EXCLUSIVAMENTE en formato JSON:

{{
  "1": {{
"RISK_SEGMENT": "...", "ALERT_LEVEL": <1-3>, "RECOMMENDED_ACTION": "<BLOCK|REVIEW|ALLOW>"}},
  "2": {{...}},
  ...
}}
"""
    return textwrap.dedent(prompt).strip()


Aplicando la función de llamada al LLM sobre una muestra de 100 transacciones para no consumir demasiados tokens.

Nota: Si toma demasiado tiempo la ejecución puede modificar la siguiente celda para solo una muestra de 10

In [ ]:
%%time
# Tomamos hasta 100 filas
df_batch = df.sample(100, random_state=42).copy()

prompt = build_batch_prompt(df_batch)

resp = model.generate_content(prompt)
text = resp.text

# Extraer JSON
start = text.find("{")
end = text.rfind("}")
json_str = text[start:end+1]
data = json.loads(json_str)  # dict: "1" -> dict con scores

# Mapear resultados a un DataFrame
rows = []
df_batch_reset = df_batch.reset_index(drop=True)

for i, row in df_batch_reset.iterrows():
    idx = str(i + 1)
    s = data.get(idx, {})
    rows.append({
        "RISK_SEGMENT":        s.get("RISK_SEGMENT", "UNKNOWN"),
        "ALERT_LEVEL":         int(s.get("ALERT_LEVEL", 2)),
        "RECOMMENDED_ACTION":  s.get("RECOMMENDED_ACTION", "REVIEW"),
    })

features_llm = pd.DataFrame(rows)
df_batch_llm = pd.concat([df_batch_reset, features_llm], axis=1)
df_batch_llm.head()


In [ ]:
df_batch_llm.head(10)

## Ejercicio 8
¿Qué mejoras planterias al prompt usado en los pasos anteriores?

COMPLETAR

## Ejercicio 9
Propón al menos 3 nuevas variables que podrías generar con un LLM.



COMPLETAR

## Ejercicio 10
Escribe el prompt que usarías (qué variables de entrada le darías, qué escala de salida).

COMPLETAR